In [1]:
from pinecone import Pinecone, ServerlessSpec
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client_emb = OpenAI(
    api_key=os.getenv('LOCAL_TOKEN'),
    base_url=os.getenv('LOCAL_URL')
)

def create_embeddings(texts):
    response = client_emb.embeddings.create(
        model=os.getenv('LOCAL_MODEL'),
        input=texts
    )
    
    response_dict = response.model_dump()

    return [data['embedding'] for data in response_dict['data']]

In [2]:
articles = [
    {"id": "art-01", "headline": "SpaceX Launches New Starship Rocket",
     "topic": "Science", "year": 2024,
     "keywords": ["space", "rocket", "nasa", "exploration"]},
    {"id": "art-02", "headline": "Bitcoin Reaches All-Time High as Investors Rush In",
     "topic": "Business", "year": 2024,
     "keywords": ["crypto", "bitcoin", "finance", "investment"]},
    {"id": "art-03", "headline": "Champions League Final: Real Madrid vs Manchester City",
     "topic": "Sport", "year": 2023,
     "keywords": ["football", "soccer", "champions league", "madrid"]},
    {"id": "art-04", "headline": "New AI Model Outperforms Humans in Medical Diagnosis",
     "topic": "Tech", "year": 2024,
     "keywords": ["ai", "healthcare", "machine learning", "diagnosis"]},
    {"id": "art-05", "headline": "Global Leaders Meet to Discuss Climate Change",
     "topic": "Science", "year": 2023,
     "keywords": ["climate", "environment", "policy", "emissions"]},
    {"id": "art-06", "headline": "Apple Announces Revolutionary New iPhone",
     "topic": "Tech", "year": 2024,
     "keywords": ["apple", "iphone", "smartphone", "technology"]},
    {"id": "art-07", "headline": "Stock Markets Plunge Amid Recession Fears",
     "topic": "Business", "year": 2023,
     "keywords": ["stocks", "economy", "recession", "wall street"]},
    {"id": "art-08", "headline": "Olympics 2024: USA Wins Gold in Swimming",
     "topic": "Sport", "year": 2024,
     "keywords": ["olympics", "swimming", "usa", "gold medal"]},
    {"id": "art-09", "headline": "Scientists Discover New Species in Amazon Rainforest",
     "topic": "Science", "year": 2023,
     "keywords": ["biology", "amazon", "species", "nature"]},
    {"id": "art-10", "headline": "Tesla Unveils Autonomous Robot for Home Use",
     "topic": "Tech", "year": 2024,
     "keywords": ["tesla", "robot", "automation", "ai"]},
]

In [3]:
# Часть 1
## Задача 1.1

pc = Pinecone(api_key=os.getenv('PINE_API'))

pc.delete_index('news-index')

pc.create_index(
    name='news-index',
    dimension=1024,
    spec=ServerlessSpec(
        cloud='aws',
        region='us-east-1'
    )
)

{
    "name": "news-index",
    "metric": "cosine",
    "host": "news-index-uhdd6kj.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1024,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "2025-

In [4]:
pc.list_indexes()

[
    {
        "name": "datacamp-index",
        "metric": "cosine",
        "host": "datacamp-index-uhdd6kj.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "region": "us-east-1",
                "cloud": "aws",
                "read_capacity": {
                    "mode": "OnDemand",
                    "status": {
                        "state": "Ready",
                        "current_shards": null,
                        "current_replicas": null
                    }
                }
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 1536,
        "deletion_protection": "disabled",
        "tags": null
    },
    {
        "name": "news-index",
        "metric": "cosine",
        "host": "news-index-uhdd6kj.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "region": "us-eas

In [5]:
## Задача 1.2
import tiktoken

decoder = tiktoken.encoding_for_model('text-embedding-3-small')

def create_article_text(article):
    return f'''Headline: {article['headline']}
Topic: {article['topic']}
Keywords: {', '.join(article['keywords'])}'''

articles_texts = [create_article_text(article) for article in articles]
total_token = sum(len(decoder.encode(article_text)) for article_text in articles_texts)
cost = 0.00002 * total_token/1000

print(f'''Total tokens: {total_token}
Cost: ${cost}''')

Total tokens: 263
Cost: $5.2600000000000005e-06


In [6]:
# Часть 2
## Задача 2.1

index = pc.Index('news-index')
stats = index.describe_index_stats()
print(stats)

print(f"\nРазмерность индекса: {stats['dimension']}")
print(f"Количество векторов: {stats['total_vector_count']}")
print(f"Заполненность: {stats['index_fullness']}")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '151',
                                    'content-type': 'application/json',
                                    'date': 'Mon, 16 Mar 2026 19:18:06 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '40',
                                    'x-pinecone-request-latency-ms': '39',
                                    'x-pinecone-response-duration-ms': '42'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}

Размерность индекса: 1024
Количество векторов: 0
Заполненность: 0.0


In [7]:
## Задача 2.2

try:
    error_index = pc.Index('non-existent-index')
    error_index.describe_index_stats()
except Exception as e:
    print(f'Ошибка: {type(e).__name__}: {e}')

Ошибка: NotFoundException: (404)
Reason: Not Found
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'vary': 'origin, access-control-request-method, access-control-request-headers', 'access-control-allow-origin': '*', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-10', 'x-cloud-trace-context': 'fe112006d12615858580cafc0ba5f759', 'date': 'Mon, 16 Mar 2026 19:18:06 GMT', 'server': 'Google Frontend', 'Content-Length': '93', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"NOT_FOUND","message":"Resource non-existent-index not found"},"status":404}



In [8]:
# Часть 3
## Задача 3.1

vectors = [
    {
        'id':article['id'],
        'values':create_embeddings(
            create_article_text(article)
        )[0]
    }
for article in articles]

vectors_dims = [len(vector['values']) == 1024 for vector in vectors]
print(f'''Размерность всех векторов корректна: {all(vectors_dims)}
Пример вектора: {vectors[0]}''')

Размерность всех векторов корректна: True
Пример вектора: {'id': 'art-01', 'values': [0.018660204485058784, -0.0247652530670166, -0.0037454222328960896, -0.02017473429441452, -0.01913156732916832, -0.012627716176211834, -0.008843764662742615, 0.058537717908620834, -0.12804444134235382, -0.0413559190928936, 0.030243420973420143, -0.007363994140177965, -0.01076898630708456, -0.004091900773346424, -0.03550013154745102, 0.0676548033952713, -0.11261134594678879, 0.054678961634635925, 0.028345046564936638, -0.0057794139720499516, -0.05349792167544365, 0.03439883142709732, 0.012055500410497189, 0.03509984165430069, -0.013932397589087486, 0.08456940948963165, -0.08855852484703064, 0.039811450988054276, -0.0244365893304348, 0.009902137331664562, 0.04556454345583916, 0.03553656488656998, -0.07470773160457611, -0.031918901950120926, -0.006612986791878939, -0.008885127492249012, -0.008218950591981411, 0.01875477097928524, -0.00466048764064908, 0.014025635085999966, 0.01001256424933672, 0.050435353

In [9]:
## Задача 3.2

index.upsert(
    vectors=vectors
)

print(f'Векторов в индексе: {index.describe_index_stats()['total_vector_count']}')

Векторов в индексе: 10


In [ ]:
# Часть 4
## Задача 4.1

index.upsert(
    vectors=vectors
)

print(f'Векторов в индексе: {index.describe_index_stats()['total_vector_count']}')
# Кол-во записей не изменилось так как был вызван метод upsert который обновляет записи в случае если идентификатор записи найден или добавляет в противном случае, в данный момент так как мы ничего не изменяли никаких данных в vectors следовательно id остались теми же и в таком случае новых записей не появилось

Векторов в индексе: 10


In [14]:
## Задача 4.2

vectors_with_meta = [
    {
        'id':article['id'],
        'values':create_embeddings(
            create_article_text(article)
        )[0],
        'metadata':{'topic':article['topic'], 'year':article['year']}
    }
for article in articles]

index.upsert(vectors=vectors_with_meta)

print(index.describe_index_stats())

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '184',
                                    'content-type': 'application/json',
                                    'date': 'Mon, 16 Mar 2026 19:27:29 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '41',
                                    'x-pinecone-request-latency-ms': '40',
                                    'x-pinecone-response-duration-ms': '42'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 10}},
 'storageFullness': 0.0,
 'total_vector_count': 10,
 'vector_type': 'dense'}


In [16]:
new_article = {
    "id": "art-11",
    "headline": "Ethereum Surges 30% Following Network Upgrade",
    "topic": "Business",
    "year": 2024,
    "keywords": ["ethereum", "crypto", "blockchain", "investment"]
}

new_vector = [
    {
        'id':new_article['id'],
        'values':create_embeddings(create_article_text(new_article))[0],
        'metadata':{'topic':new_article['topic'], 'year':new_article['year']}
    }
]

index.upsert(vectors=new_vector)

print(index.describe_index_stats())

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '184',
                                    'content-type': 'application/json',
                                    'date': 'Mon, 16 Mar 2026 19:29:45 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '41',
                                    'x-pinecone-request-latency-ms': '40',
                                    'x-pinecone-response-duration-ms': '42'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 11}},
 'storageFullness': 0.0,
 'total_vector_count': 11,
 'vector_type': 'dense'}


In [17]:
print(f'Векторов после добавления новой статьи: {index.describe_index_stats()['total_vector_count']}')

Векторов после добавления новой статьи: 11


In [21]:
# Часть 5
## Задача 5.1

queries = [
    "artificial intelligence and robotics",
    "financial crisis and investment",
    "olympic games and athletic achievements",
]

for query in queries:
    print(f'Запрос: "{query}"')

    results = index.query(
        vector=create_embeddings(query)[0],
        top_k=3,
        include_metadata=True
    )

    for i, match in enumerate(results['matches']):
        print(f'    {i+1}. {match['id']} | {round(match['score'], 4)} | metadata: {match['metadata']}')

Запрос: "artificial intelligence and robotics"
    1. art-10 | 0.592 | metadata: {'topic': 'Tech', 'year': 2024}
    2. art-04 | 0.5063 | metadata: {'topic': 'Tech', 'year': 2024}
    3. art-01 | 0.3486 | metadata: {'topic': 'Science', 'year': 2024}
Запрос: "financial crisis and investment"
    1. art-07 | 0.5329 | metadata: {'topic': 'Business', 'year': 2023}
    2. art-02 | 0.4874 | metadata: {'topic': 'Business', 'year': 2024}
    3. art-11 | 0.4179 | metadata: {'topic': 'Business', 'year': 2024}
Запрос: "olympic games and athletic achievements"
    1. art-08 | 0.6005 | metadata: {'topic': 'Sport', 'year': 2024}
    2. art-03 | 0.3762 | metadata: {'topic': 'Sport', 'year': 2023}
    3. art-01 | 0.297 | metadata: {'topic': 'Science', 'year': 2024}


In [ ]:
## Задача 5.2

queries = [
    'technology and innovation',
    'science discoveries',
    'sports championships'
]

print('=== Только Tech ===')
results = index.query(
    vector=create_embeddings(queries[0])[0],
    top_k=3,
    include_metadata=True,
    filter={'topic':{'$eq':'Tech'}}
)

for i, match in enumerate(results['matches']):
    print(f'    {i+1}. {match['id']} | score={round(match['score'],4)}')

print('\n=== Только 2023 год ===')
results = index.query(
    vector=create_embeddings(queries[1])[0],
    top_k=3,
    include_metadata=True,
    filter={'year':{'$eq':2023}}
)

for i, match in enumerate(results['matches']):
    print(f'    {i+1}. {match['id']} | score={round(match['score'],4)}')

print('\n=== Исключая Business ===')
results = index.query(
    vector=create_embeddings(queries[2])[0],
    top_k=3,
    include_metadata=True,
    filter={'topic':{'$ne':'Business'}}
)

for i, match in enumerate(results['matches']):
    print(f'    {i+1}. {match['id']} | score={round(match['score'],4)}')

=== Только Tech ===
    1. art-06 | score=0.5291
    2. art-10 | score=0.5061
    3. art-04 | score=0.4979

=== Только 2023 год ===
    1. art-09 | score=0.5676
    2. art-05 | score=0.4656
    3. art-03 | score=0.3069

=== Исключая Business ===
    1. art-08 | score=0.5578
    2. art-03 | score=0.4973
    3. art-01 | score=0.3317


In [41]:
articles = [
    {"id": "art-01", "headline": "SpaceX Launches New Starship Rocket",
     "topic": "Science", "year": 2024,
     "keywords": ["space", "rocket", "nasa", "exploration"]},
    {"id": "art-02", "headline": "Bitcoin Reaches All-Time High as Investors Rush In",
     "topic": "Business", "year": 2024,
     "keywords": ["crypto", "bitcoin", "finance", "investment"]},
    {"id": "art-03", "headline": "Champions League Final: Real Madrid vs Manchester City",
     "topic": "Sport", "year": 2023,
     "keywords": ["football", "soccer", "champions league", "madrid"]},
    {"id": "art-04", "headline": "New AI Model Outperforms Humans in Medical Diagnosis",
     "topic": "Tech", "year": 2024,
     "keywords": ["ai", "healthcare", "machine learning", "diagnosis"]},
    {"id": "art-05", "headline": "Global Leaders Meet to Discuss Climate Change",
     "topic": "Science", "year": 2023,
     "keywords": ["climate", "environment", "policy", "emissions"]},
    {"id": "art-06", "headline": "Apple Announces Revolutionary New iPhone",
     "topic": "Tech", "year": 2024,
     "keywords": ["apple", "iphone", "smartphone", "technology"]},
    {"id": "art-07", "headline": "Stock Markets Plunge Amid Recession Fears",
     "topic": "Business", "year": 2023,
     "keywords": ["stocks", "economy", "recession", "wall street"]},
    {"id": "art-08", "headline": "Olympics 2024: USA Wins Gold in Swimming",
     "topic": "Sport", "year": 2024,
     "keywords": ["olympics", "swimming", "usa", "gold medal"]},
    {"id": "art-09", "headline": "Scientists Discover New Species in Amazon Rainforest",
     "topic": "Science", "year": 2023,
     "keywords": ["biology", "amazon", "species", "nature"]},
    {"id": "art-10", "headline": "Tesla Unveils Autonomous Robot for Home Use",
     "topic": "Tech", "year": 2024,
     "keywords": ["tesla", "robot", "automation", "ai"]},
     {
    "id": "art-11",
    "headline": "Ethereum Surges 30% Following Network Upgrade",
    "topic": "Business",
    "year": 2024,
    "keywords": ["ethereum", "crypto", "blockchain", "investment"]
}
]

In [42]:
## Задача 5.3

user_history = [
    {"id": "art-02", "headline": "Bitcoin Reaches All-Time High as Investors Rush In",
     "topic": "Business", "year": 2024,
     "keywords": ["crypto", "bitcoin", "finance", "investment"]}
]

user_text = [create_article_text(article) for article in user_history]
user_values = create_embeddings(user_text)[0]

print(f'Эталонная статья: "{user_history[0]['headline']}"\n')

result = index.query(
    vector=user_values,
    top_k=4,
    include_metadata=True,
    filter={'topic':'Business'}
)

print('Рекомендации (только Business)')
rank = 1
for i, match in enumerate(result['matches']):
    if match['id'] == user_history[0]['id']:
        continue
    headline = next((a['headline'] for a in articles if a['id'] == match['id']), match['id'])
    print(f"  {rank}. {match['id']} | {headline} | score={match['score']:.4f}")
    rank += 1

Эталонная статья: "Bitcoin Reaches All-Time High as Investors Rush In"

Рекомендации (только Business)
  1. art-11 | Ethereum Surges 30% Following Network Upgrade | score=0.7224
  2. art-07 | Stock Markets Plunge Amid Recession Fears | score=0.5036


In [43]:
## Задача 6

tech_vectors = [v for v, a in zip(vectors_with_meta, articles) if a['topic'] == 'Tech']
business_vectors = [v for v, a in zip(vectors_with_meta, articles) if a['topic'] == 'Business']

index.upsert(vectors=tech_vectors, namespace='tech-namespace')
index.upsert(vectors=business_vectors, namespace='business-namespace')

stats = index.describe_index_stats()
print('Namespaces:', stats['namespaces'])

result = index.query(
    vector=create_embeddings('artificial intelligence')[0],
    top_k=3,
    include_metadata=True,
    namespace='tech-namespace'
)
print('\nРезультаты в tech-namespace:')
for match in result['matches']:
    print(f"  {match['id']} | score={match['score']:.4f}")

Namespaces: {'tech-namespace': {'vector_count': 3}, 'business-namespace': {'vector_count': 2}, '__default__': {'vector_count': 11}}

Результаты в tech-namespace:
  art-04 | score=0.5255
  art-06 | score=0.3541
  art-10 | score=0.4757


In [44]:
pc.delete_index('news-index')
print('Индекс удалён.')
print('Оставшиеся индексы:', pc.list_indexes())

Индекс удалён.
Оставшиеся индексы: [{
    "name": "datacamp-index",
    "metric": "cosine",
    "host": "datacamp-index-uhdd6kj.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null
}]
